# Aggregate Metrics

Stickler automatically includes an `aggregate` field at every node in the confusion-matrix result tree. This notebook walks through two examples that cover all four node types:

1. **Primitive + List of Primitives + Nested Structure** — shows how leaf and parent aggregates differ.
2. **List of StructuredModel** — shows that aggregate metrics recurse into fields even when the object-level classification is FD or FN.

## Setup

In [1]:
import json
from typing import List

from stickler.comparators.exact import ExactComparator
from stickler.comparators.levenshtein import LevenshteinComparator
from stickler.structured_object_evaluator.models.comparable_field import ComparableField
from stickler.structured_object_evaluator.models.structured_model import StructuredModel


def pp(obj):
    """Pretty-print a dict as JSON."""
    print(json.dumps(obj, indent=2))

## Example 1: Primitive + List of Primitives + Nested Structure

This model has three node types:
- `name` — primitive (leaf)
- `tags` — `List[str]` (leaf, matched via Hungarian algorithm)
- `contact` — nested `StructuredModel` (parent)

In [2]:
class Contact(StructuredModel):
    phone: str = ComparableField(comparator=ExactComparator(), threshold=1.0)
    email: str = ComparableField(comparator=ExactComparator(), threshold=1.0)


class Person(StructuredModel):
    name: str = ComparableField(comparator=ExactComparator(), threshold=1.0)
    tags: List[str] = ComparableField(comparator=ExactComparator(), threshold=1.0)
    contact: Contact = ComparableField(comparator=ExactComparator(), threshold=1.0)

In [3]:
gt = Person(
    name="John",
    tags=["vip", "active", "premium"],
    contact=Contact(phone="123", email="john@test.com"),
)
pred = Person(
    name="John",
    tags=["vip", "premium"],  # missing "active"
    contact=Contact(phone="456", email="john@test.com"),  # wrong phone
)

result = gt.compare_with(pred, include_confusion_matrix=True)
cm = result["confusion_matrix"]

### Top-level aggregate

Sums all leaf-level counts: name (1 TP) + tags (2 TP, 1 FN) + phone (1 FD) + email (1 TP) = 4 TP, 1 FD, 1 FN.

In [4]:
pp(cm["aggregate"])

{
  "tp": 4,
  "fa": 0,
  "fd": 1,
  "fp": 1,
  "tn": 0,
  "fn": 1,
  "derived": {
    "cm_precision": 0.8,
    "cm_recall": 0.8,
    "cm_f1": 0.8000000000000002,
    "cm_accuracy": 0.6666666666666666
  }
}


### Field-level aggregates

- `name`: primitive leaf — `aggregate` equals `overall`.
- `tags`: `List[str]` leaf — Hungarian matching gives 2 TP ("vip", "premium") and 1 FN ("active"). `aggregate` equals `overall`.
- `contact`: parent node — `overall` is FD (phone mismatch drags it down), but `aggregate` sums children: 1 TP (email) + 1 FD (phone).

In [5]:
for field_name in ["name", "tags", "contact"]:
    field = cm["fields"][field_name]
    print(f"--- {field_name} ---")
    print(f"  overall:   {field['overall']}")
    print(f"  aggregate: {field['aggregate']}")
    print()

--- name ---
  overall:   {'tp': 1, 'fa': 0, 'fd': 0, 'fp': 0, 'tn': 0, 'fn': 0, 'derived': {'cm_precision': 1.0, 'cm_recall': 1.0, 'cm_f1': 1.0, 'cm_accuracy': 1.0}}
  aggregate: {'tp': 1, 'fa': 0, 'fd': 0, 'fp': 0, 'tn': 0, 'fn': 0, 'derived': {'cm_precision': 1.0, 'cm_recall': 1.0, 'cm_f1': 1.0, 'cm_accuracy': 1.0}}

--- tags ---
  overall:   {'tp': 2, 'fa': 0, 'fd': 0, 'fp': 0, 'tn': 0, 'fn': 1, 'derived': {'cm_precision': 1.0, 'cm_recall': 0.6666666666666666, 'cm_f1': 0.8, 'cm_accuracy': 0.6666666666666666}}
  aggregate: {'tp': 2, 'fa': 0, 'fd': 0, 'fp': 0, 'tn': 0, 'fn': 1, 'derived': {'cm_precision': 1.0, 'cm_recall': 0.6666666666666666, 'cm_f1': 0.8, 'cm_accuracy': 0.6666666666666666}}

--- contact ---
  overall:   {'tp': 0, 'fa': 0, 'fd': 1, 'fp': 1, 'tn': 0, 'fn': 0, 'similarity_score': 0.5, 'all_fields_matched': False, 'derived': {'cm_precision': 0.0, 'cm_recall': 0.0, 'cm_f1': 0.0, 'cm_accuracy': 0.0}}
  aggregate: {'tp': 1, 'fa': 0, 'fd': 1, 'fp': 1, 'tn': 0, 'fn': 0, 'der

## Example 2: List of StructuredModel — FD Recursion and Unmatched Items

This example illustrates two key behaviors:
1. An object pair classified as **FD** (below `match_threshold`) still has its fields recursed for aggregate metrics.
2. An unmatched GT item (**FN**) contributes its populated fields to the aggregate.

In [6]:
class LineItem(StructuredModel):
    match_threshold = 0.6

    sku: str = ComparableField(comparator=ExactComparator(), threshold=1.0, weight=2.0)
    description: str = ComparableField(
        comparator=LevenshteinComparator(), threshold=0.7, weight=1.0
    )
    qty: int = ComparableField(comparator=ExactComparator(), threshold=1.0, weight=1.0)


class Invoice(StructuredModel):
    invoice_id: str = ComparableField(comparator=ExactComparator(), threshold=1.0)
    items: List[LineItem] = ComparableField(weight=1.0)

In [7]:
gt = Invoice(
    invoice_id="INV-001",
    items=[
        LineItem(sku="AAA", description="Widget", qty=10),
        LineItem(sku="BBB", description="Gadget", qty=5),
        LineItem(sku="CCC", description="Cable", qty=2),  # no pred counterpart
    ],
)
pred = Invoice(
    invoice_id="INV-001",
    items=[
        LineItem(sku="AAA", description="Widget", qty=10),            # TP (sim 1.0)
        LineItem(sku="BBB", description="Completely Wrong", qty=99),  # FD (sim 0.53)
    ],
)

result2 = gt.compare_with(pred, include_confusion_matrix=True)
cm2 = result2["confusion_matrix"]

### Object-level vs aggregate for `items`

- `overall`: 1 TP (AAA, sim 1.0 >= 0.6), 1 FD (BBB, sim 0.53 < 0.6), 1 FN (CCC, unmatched).
- Even though BBB is FD at the object level, its fields are still recursed for aggregate purposes.

| Item | Object classification | `sku` | `description` | `qty` |
|------|----------------------|-------|---------------|-------|
| AAA pair | TP (sim 1.0 >= 0.6) | TP (exact match) | TP (exact match) | TP (exact match) |
| BBB pair | FD (sim 0.53 < 0.6) | TP (exact match) | FD (low similarity) | FD (5 ≠ 99) |
| CCC | FN (unmatched) | FN | FN | FN |

Summing the columns gives `items.aggregate`: 4 TP, 2 FD, 3 FN.

In [8]:
items = cm2["fields"]["items"]
print("items.overall:")
pp(items["overall"])
print("\nitems.aggregate:")
pp(items["aggregate"])

items.overall:
{
  "tp": 1,
  "fa": 0,
  "fd": 1,
  "fp": 1,
  "tn": 0,
  "fn": 1,
  "derived": {
    "cm_precision": 0.5,
    "cm_recall": 0.5,
    "cm_f1": 0.5,
    "cm_accuracy": 0.3333333333333333
  }
}

items.aggregate:
{
  "tp": 4,
  "fa": 0,
  "fd": 2,
  "fp": 2,
  "tn": 0,
  "fn": 3,
  "derived": {
    "cm_precision": 0.6666666666666666,
    "cm_recall": 0.5714285714285714,
    "cm_f1": 0.6153846153846153,
    "cm_accuracy": 0.4444444444444444
  }
}


### Per-sub-field breakdown across all matched and unmatched items

In [9]:
for field_name, field_data in items["fields"].items():
    agg = field_data["aggregate"]
    print(f"{field_name}: TP={agg['tp']}  FD={agg['fd']}  FN={agg['fn']}")

sku: TP=2  FD=0  FN=1
description: TP=1  FD=1  FN=1
qty: TP=1  FD=1  FN=1


### Full confusion matrix tree

In [10]:
pp(cm2)

{
  "overall": {
    "tp": 2,
    "fa": 0,
    "fd": 1,
    "fp": 1,
    "tn": 0,
    "fn": 1,
    "similarity_score": 0.75,
    "all_fields_matched": true,
    "derived": {
      "cm_precision": 0.6666666666666666,
      "cm_recall": 0.6666666666666666,
      "cm_f1": 0.6666666666666666,
      "cm_accuracy": 0.5
    }
  },
  "fields": {
    "invoice_id": {
      "overall": {
        "tp": 1,
        "fa": 0,
        "fd": 0,
        "fp": 0,
        "tn": 0,
        "fn": 0,
        "derived": {
          "cm_precision": 1.0,
          "cm_recall": 1.0,
          "cm_f1": 1.0,
          "cm_accuracy": 1.0
        }
      },
      "raw_similarity_score": 1.0,
      "similarity_score": 1.0,
      "threshold_applied_score": 1.0,
      "weight": 1.0,
      "aggregate": {
        "tp": 1,
        "fa": 0,
        "fd": 0,
        "fp": 0,
        "tn": 0,
        "fn": 0,
        "derived": {
          "cm_precision": 1.0,
          "cm_recall": 1.0,
          "cm_f1": 1.0,
          "cm_a

### Hierarchical reporting helper

A simple recursive function to print aggregate precision / recall / F1 at every level.

In [11]:
def print_metrics(node, path=""):
    if "aggregate" in node:
        a = node["aggregate"]
        d = a.get("derived") or {}
        p = d.get("cm_precision", 0)
        r = d.get("cm_recall", 0)
        f1 = d.get("cm_f1", 0)
        print(f"{path or 'root':30s}  P={p:.3f}  R={r:.3f}  F1={f1:.3f}")
    for name, child in node.get("fields", {}).items():
        print_metrics(child, f"{path}.{name}" if path else name)


print_metrics(cm2)

root                            P=0.714  R=0.625  F1=0.667
invoice_id                      P=1.000  R=1.000  F1=1.000
items                           P=0.667  R=0.571  F1=0.615
items.sku                       P=1.000  R=0.667  F1=0.800
items.description               P=0.500  R=0.500  F1=0.500
items.qty                       P=0.500  R=0.500  F1=0.500
